# Feature Engineering

In [68]:
from pathlib import Path

import numpy as np
import pandas as pd

model_ready_data = pd.read_csv("../data/processed/model_ready_hourly_rental_data.csv")
model_ready_data["hour"] = pd.to_datetime(model_ready_data["hour"])
model_ready_data["date"] = pd.to_datetime(model_ready_data["date"])

model_ready_data = (
    model_ready_data
    .sort_values("hour")
    .reset_index(drop=True)
)

## 0. Fill missing hourly timestamps with weekday + hour average

In [69]:
# missing hours
expected_hours = pd.date_range(
    start=model_ready_data["hour"].min(),
    end=model_ready_data["hour"].max(),
    freq="h",
)
len(expected_hours.difference(model_ready_data["hour"]))

165

In [70]:
engineered_data = (
    model_ready_data
    .set_index("hour")
    .sort_index()
)

full_hour_index = pd.date_range(
    start=engineered_data.index.min(),
    end=engineered_data.index.max(),
    freq="h",
)

engineered_data = engineered_data.reindex(full_hour_index)

engineered_data.index.name = "hour"
engineered_data = engineered_data.reset_index()

In [71]:
engineered_data["date"] = engineered_data["hour"].dt.date
engineered_data["hour_of_day"] = engineered_data["hour"].dt.hour
engineered_data["day_of_week"] = engineered_data["hour"].dt.dayofweek
engineered_data["month"] = engineered_data["hour"].dt.month
engineered_data["is_weekend"] = (engineered_data["day_of_week"] >= 5).astype(int)
engineered_data["was_missing_hour"] = engineered_data["total_count"].isna().astype(int)

In [72]:
COUNT_COLUMNS = [
    "direct_count",
    "registered_count",
    "total_count",
]

# average of previous weekday + hour
for col in COUNT_COLUMNS:
    group_mean = (
        engineered_data
        .groupby(["day_of_week", "hour_of_day"])[col]
        .transform("mean")
    )
    
    engineered_data[col] = engineered_data[col].fillna(group_mean)

# average of hour of the day
for col in COUNT_COLUMNS:
    hour_mean = (
        engineered_data
        .groupby("hour_of_day")[col]
        .transform("mean")
    )
    
    engineered_data[col] = engineered_data[col].fillna(hour_mean)

# global average
for col in COUNT_COLUMNS:
    engineered_data[col] = engineered_data[col].fillna(engineered_data[col].mean())

In [73]:
# fill weather
WEATHER_NUMERIC_COLUMNS = [
    "temperature_c",
    "perceived_temperature_c",
    "humidity",
    "windspeed_kmh",
]

engineered_data[WEATHER_NUMERIC_COLUMNS] = (
    engineered_data[WEATHER_NUMERIC_COLUMNS]
    .interpolate(method="linear")
    .ffill()
    .bfill()
)

engineered_data["conditions"] = (
    engineered_data["conditions"]
    .ffill()
    .bfill()
)

In [74]:
# create day types
engineered_data["is_holiday"] = engineered_data["is_holiday"].fillna(0).astype(int)

engineered_data["is_workday"] = (
    (engineered_data["day_of_week"] < 5)
    & (engineered_data["is_holiday"] == 0)
).astype(int)

## 1. Create calendar-based features

In [75]:
# rush_hour
engineered_data["rush_hour"] = engineered_data["hour_of_day"].isin(
    [7, 8, 9, 16, 17, 18]
).astype(int)

# season
def get_season(month):
    if month in [12, 1, 2]:
        return "winter"
    elif month in [3, 4, 5]:
        return "spring"
    elif month in [6, 7, 8]:
        return "summer"
    else:
        return "autumn"

engineered_data["season"] = engineered_data["month"].apply(get_season)


# cyclical encoding for: hour, weekdays and month
engineered_data["hour_sin"] = np.sin(2 * np.pi * engineered_data["hour_of_day"] / 24)
engineered_data["hour_cos"] = np.cos(2 * np.pi * engineered_data["hour_of_day"] / 24)

engineered_data["weekday_sin"] = np.sin(2 * np.pi * engineered_data["day_of_week"] / 7)
engineered_data["weekday_cos"] = np.cos(2 * np.pi * engineered_data["day_of_week"] / 7)

engineered_data["month_sin"] = np.sin(2 * np.pi * (engineered_data["month"] - 1) / 12)
engineered_data["month_cos"] = np.cos(2 * np.pi * (engineered_data["month"] - 1) / 12)

engineered_data[
    [
        "hour_of_day",
        "hour_sin",
        "hour_cos",
        "day_of_week",
        "weekday_sin",
        "weekday_cos",
        "month",
        "month_sin",
        "month_cos",
    ]
].head(24)

,hour_of_day,hour_sin,hour_cos,day_of_week,weekday_sin,weekday_cos,month,month_sin,month_cos
0,0,0.000000e+00,1.000000e+00,5,-0.974928,-0.222521,1,0.0,1.0
1,1,2.588190e-01,9.659258e-01,5,-0.974928,-0.222521,1,0.0,1.0
2,2,5.000000e-01,8.660254e-01,5,-0.974928,-0.222521,1,0.0,1.0
3,3,7.071068e-01,7.071068e-01,5,-0.974928,-0.222521,1,0.0,1.0
4,4,8.660254e-01,5.000000e-01,5,-0.974928,-0.222521,1,0.0,1.0
5,5,9.659258e-01,2.588190e-01,5,-0.974928,-0.222521,1,0.0,1.0
6,6,1.000000e+00,6.123234e-17,5,-0.974928,-0.222521,1,0.0,1.0
7,7,9.659258e-01,-2.588190e-01,5,-0.974928,-0.222521,1,0.0,1.0
8,8,8.660254e-01,-5.000000e-01,5,-0.974928,-0.222521,1,0.0,1.0
9,9,7.071068e-01,-7.071068e-01,5,-0.974928,-0.222521,1,0.0,1.0


## 2. Historical demand features
### 2.1 Lag features

In [76]:
engineered_data["total_lag_24h"] = engineered_data["total_count"].shift(24)

engineered_data["total_lag_7d"] = engineered_data["total_count"].shift(24 * 7)

### 2.2 Rolling average features

In [77]:
engineered_data["total_24h_mean"] = (
    engineered_data["total_count"]
    .shift(1) # avoid data leakage
    .rolling(window=24)
    .mean()
)

engineered_data["total_7d_mean"] = (
    engineered_data["total_count"]
    .shift(1)
    .rolling(window=24 * 7)
    .mean()
)

In [78]:
#Same day same hour preovious 4w average
engineered_data["total_count_lag_1w"] = (
    engineered_data["total_count"].shift(24 * 7)
)

engineered_data["total_count_lag_2w"] = (
    engineered_data["total_count"].shift(24 * 7 * 2)
)

engineered_data["total_count_lag_3w"] = (
    engineered_data["total_count"].shift(24 * 7 * 3)
)

engineered_data["total_count_lag_4w"] = (
    engineered_data["total_count"].shift(24 * 7 * 4)
)

engineered_data["same_hour_weekday_4w_mean"] = engineered_data[
    [
        "total_count_lag_1w",
        "total_count_lag_2w",
        "total_count_lag_3w",
        "total_count_lag_4w",
    ]
 ].mean(axis=1)              

In [79]:
engineered_data.isna().sum().sort_values(ascending=False)

total_count_lag_4w           672
total_count_lag_3w           504
total_count_lag_2w           336
same_hour_weekday_4w_mean    168
total_lag_7d                 168
total_count_lag_1w           168
total_7d_mean                168
holiday                      165
total_lag_24h                 24
total_24h_mean                24
rush_hour                      0
month_cos                      0
month_sin                      0
weekday_cos                    0
weekday_sin                    0
hour_cos                       0
hour_sin                       0
season                         0
hour                           0
direct_count                   0
is_workday                     0
is_holiday                     0
windspeed_kmh                  0
humidity                       0
perceived_temperature_c        0
temperature_c                  0
conditions                     0
is_weekend                     0
month                          0
day_of_week                    0
hour_of_da

## 3. Extract final features 

In [83]:
TARGET = "total_count"

NUMERIC_FEATURES = [
    # weather
    "temperature_c",
    "perceived_temperature_c",
    "humidity",
    "windspeed_kmh",

    # calendar
    "is_weekend",
    "is_holiday",
    "is_workday",
    "rush_hour",

    # cyclical time features
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",

    # historical demand features
    "total_lag_24h",
    "total_lag_7d",
    "total_24h_mean",
    "total_7d_mean",
    "same_hour_weekday_4w_mean",
]

CATEGORICAL_FEATURES = [
    "conditions",
    "season",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

engineered_model_data = engineered_data[
    ["hour", *FEATURES, TARGET]
].copy()

engineered_model_data = (
    engineered_model_data
    .dropna(subset=FEATURES + [TARGET])
    .reset_index(drop=True)
)

engineered_model_data.head()

,hour,temperature_c,perceived_temperature_c,humidity,windspeed_kmh,is_weekend,is_holiday,is_workday,rush_hour,hour_sin,...,month_sin,month_cos,total_lag_24h,total_lag_7d,total_24h_mean,total_7d_mean,same_hour_weekday_4w_mean,conditions,season,total_count
0,2011-01-08 00:00:00,0.5,-3.0,51.0,11.0,1,0,0,0,0.000000,...,0.0,1.0,17.000000,16.0,63.191585,56.296613,16.0,clouds,winter,25.0
1,2011-01-08 01:00:00,0.5,-2.0,55.0,6.0,1,0,0,0,0.258819,...,0.0,1.0,7.000000,40.0,63.524918,56.350184,40.0,clouds,winter,16.0
2,2011-01-08 02:00:00,0.5,-0.0,55.0,0.0,1,0,0,0,0.500000,...,0.0,1.0,1.000000,32.0,63.899918,56.207327,32.0,clouds,winter,16.0
3,2011-01-08 03:00:00,0.5,-3.0,55.0,11.0,1,0,0,0,0.707107,...,0.0,1.0,6.598039,13.0,64.524918,56.112089,13.0,light_rain,winter,7.0
4,2011-01-08 04:00:00,0.5,-3.0,55.0,11.0,1,0,0,0,0.866025,...,0.0,1.0,1.000000,1.0,64.541667,56.076375,1.0,light_rain,winter,1.0


In [84]:
engineered_model_data.isna().sum().sort_values(ascending=False)

hour                         0
weekday_cos                  0
season                       0
conditions                   0
same_hour_weekday_4w_mean    0
total_7d_mean                0
total_24h_mean               0
total_lag_7d                 0
total_lag_24h                0
month_cos                    0
month_sin                    0
weekday_sin                  0
temperature_c                0
hour_cos                     0
hour_sin                     0
rush_hour                    0
is_workday                   0
is_holiday                   0
is_weekend                   0
windspeed_kmh                0
humidity                     0
perceived_temperature_c      0
total_count                  0
dtype: int64